# 04 · Build the SQLite star schema and the Power BI tables

**Goal:** load everything into `02_Database/geo_inr_analytics.db` (a star schema with foreign keys), run the final
data-quality checks, and export one CSV per table to `01_Data/4_PowerBI_Tables/` for Power BI.

> **Series:** 01 Ingest → 02 Align & transform → 03 Event windows → 04 Database → 05 Results report → 06 Analysis & charts. Run them in order (or run `python run_pipeline.py` from the project folder). Paths and settings live in `config.py`.

In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import pandas as pd
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

import shutil
import sqlite3
import tempfile
from pathlib import Path
import numpy as np
import pandas as pd
from config import (PROC, REF, PBI, REF_EVENTS, REF_MILESTONES, REF_CATALOG, DB_PATH, WINDOWS, MAX_STALENESS_DAYS, ANALYSIS_START,
                    ANALYSIS_END, FX_SOURCES, read_reference)

## Schema

- **Dimensions:** date, event, milestone, window, series, analysis series, FX basis.
- **Facts:** native-frequency data (FX, commodities, markets, macro), the aligned daily panel, and the event-window tables.
- **Data quality:** the issue log, series coverage, alignment and the FX cross-check.

The DDL declares primary and foreign keys, so a row pointing at a series or event that doesn't exist fails to load.
That check caught a missing catalogue entry during development.

In [2]:
DDL = """
PRAGMA foreign_keys = ON;

CREATE TABLE dim_date (
  date_key INTEGER PRIMARY KEY, date TEXT NOT NULL, year INTEGER, quarter TEXT,
  month_num INTEGER, month_name TEXT, year_month TEXT, day_of_week TEXT, is_weekend INTEGER,
  fiscal_year_in TEXT, is_rbi_fx_day INTEGER, is_us_h10_day INTEGER, rbi_listed_holiday INTEGER);

CREATE TABLE dim_series (
  series_id TEXT PRIMARY KEY, series_name TEXT, fact_table TEXT, category TEXT, unit TEXT,
  frequency TEXT, source_org TEXT, source_dataset TEXT, source_code TEXT, source_url TEXT,
  native_timestamp TEXT, role_in_analysis TEXT, notes TEXT);

CREATE TABLE dim_fx_basis (
  basis TEXT PRIMARY KEY, fx_series_id TEXT REFERENCES dim_series(series_id),
  calendar_definition TEXT, is_primary INTEGER);

CREATE TABLE dim_event (
  event_id TEXT PRIMARY KEY, event_name TEXT NOT NULL, short_name TEXT, event_type TEXT, affected_region TEXT,
  india_direct TEXT, event_start_date TEXT NOT NULL, event_end_date TEXT, end_date_basis TEXT,
  anchor_date TEXT NOT NULL, anchor_rule TEXT, timing_note TEXT, source_org TEXT,
  source_title TEXT, source_url TEXT, source_pub_date TEXT, source_type TEXT,
  selection_rationale TEXT, known_concurrent_factors TEXT, duration_days INTEGER,
  event_label TEXT);

CREATE TABLE dim_event_milestone (
  event_id TEXT REFERENCES dim_event(event_id), milestone_date TEXT, milestone TEXT,
  source_org TEXT, source_url TEXT, source_pub_date TEXT, source_type TEXT, date_key INTEGER);

CREATE TABLE dim_analysis_series (
  series_id TEXT PRIMARY KEY, display_name TEXT, unit TEXT, change_unit TEXT, sort_order INTEGER,
  source_series_rbi_basis TEXT, source_series_fred_basis TEXT, interpretation_of_positive TEXT);

CREATE TABLE dim_window (
  window_id TEXT PRIMARY KEY, window_label TEXT, start_offset INTEGER, end_offset INTEGER,
  window_type TEXT, sort_order INTEGER, definition TEXT);

CREATE TABLE fact_fx (
  date_key INTEGER REFERENCES dim_date(date_key), series_id TEXT REFERENCES dim_series(series_id),
  usd_inr REAL, usdinr_open REAL, usdinr_close REAL, inr_strongest REAL, inr_weakest REAL,
  usdinr_pct_change REAL, inr_pct_change REAL, log_return REAL, days_since_prev_obs INTEGER,
  PRIMARY KEY (date_key, series_id));

CREATE TABLE fact_commodity (
  date_key INTEGER REFERENCES dim_date(date_key), series_id TEXT REFERENCES dim_series(series_id),
  value REAL, pct_change REAL, log_return REAL, days_since_prev_obs INTEGER,
  PRIMARY KEY (date_key, series_id));

CREATE TABLE fact_market (
  date_key INTEGER REFERENCES dim_date(date_key), series_id TEXT REFERENCES dim_series(series_id),
  value REAL, change REAL, change_unit TEXT, log_return REAL, days_since_prev_obs INTEGER,
  PRIMARY KEY (date_key, series_id));

CREATE TABLE fact_macro (
  date_key INTEGER, series_id TEXT REFERENCES dim_series(series_id), period_start TEXT,
  frequency TEXT, value REAL, yoy_pct REAL, mom_change REAL, series_version TEXT,
  PRIMARY KEY (date_key, series_id));

CREATE TABLE fact_daily_aligned (
  basis TEXT REFERENCES dim_fx_basis(basis), date_key INTEGER REFERENCES dim_date(date_key),
  trading_day_seq INTEGER, usdinr REAL, usdinr_pct REAL, inr_pct REAL, usdinr_logret REAL,
  brent REAL, gold_usd REAL, nifty50 REAL, dxy_broad REAL, vix REAL, ust10y REAL,
  ffr_upper REAL, in_repo REAL, rate_diff_in_us REAL,
  brent_logret REAL, gold_logret REAL, nifty_logret REAL, dxy_logret REAL, vix_logret REAL,
  ust10y_change_pp REAL,
  usdinr_vol20 REAL, brent_vol20 REAL, gold_vol20 REAL, nifty_vol20 REAL, dxy_vol20 REAL,
  corr60_brent REAL, corr60_gold REAL, corr60_nifty REAL, corr60_dxy REAL, corr60_vix REAL,
  corr60_ust10y REAL,
  PRIMARY KEY (basis, date_key));

CREATE TABLE fact_event_anchor (
  basis TEXT, event_id TEXT REFERENCES dim_event(event_id), covered INTEGER, anchor_date TEXT,
  t0_date TEXT, t_minus1_date TEXT, calendar_days_anchor_to_t0 INTEGER,
  last_available_date TEXT, trading_days_after_available INTEGER, note TEXT,
  t0_date_key INTEGER, PRIMARY KEY (basis, event_id));

CREATE TABLE fact_event_window (
  basis TEXT, event_id TEXT REFERENCES dim_event(event_id),
  window_id TEXT REFERENCES dim_window(window_id), series_id TEXT, status TEXT,
  start_date TEXT, end_date TEXT, trading_days INTEGER, start_value REAL, end_value REAL,
  change REAL, log_change REAL, change_unit TEXT, n_returns INTEGER, window_vol_ann_pct REAL,
  start_staleness_days REAL, end_staleness_days REAL, inr_pct_change REAL,
  usdinr_minus_dxy_pp REAL, baseline_n INTEGER, baseline_mean_pct REAL, baseline_sd_pct REAL,
  z_score REAL, percentile REAL, direction TEXT, is_unusual_2sd INTEGER,
  PRIMARY KEY (basis, event_id, window_id, series_id));

CREATE TABLE fact_event_window_corr (
  basis TEXT, event_id TEXT, window_id TEXT, series_id TEXT, n_obs INTEGER, status TEXT,
  corr REAL, PRIMARY KEY (basis, event_id, window_id, series_id));

CREATE TABLE fact_event_path (
  basis TEXT, event_id TEXT, rel_day INTEGER, date TEXT, series_id TEXT, indexed_value REAL,
  PRIMARY KEY (basis, event_id, rel_day, series_id));

CREATE TABLE fact_event_window_context (
  basis TEXT, event_id TEXT, window_id TEXT, start_date TEXT, end_date TEXT, status TEXT,
  trading_days INTEGER, overlapping_events TEXT, rbi_repo_changes TEXT,
  fomc_target_changes TEXT, rbi_net_fx_usd_mn REAL, rbi_fx_months TEXT,
  n_overlapping INTEGER, has_policy_change INTEGER,
  PRIMARY KEY (basis, event_id, window_id));

CREATE TABLE dq_issues (
  issue_id INTEGER PRIMARY KEY, stage TEXT, series_id TEXT, obs_date TEXT, rule TEXT,
  detail TEXT, severity TEXT, action TEXT);
CREATE TABLE dq_series_coverage (
  series_id TEXT PRIMARY KEY, first_date TEXT, last_date TEXT, n_obs INTEGER,
  expected_frequency TEXT, max_gap_calendar_days INTEGER, gaps_over_5d INTEGER,
  days_behind_build_date INTEGER, status TEXT);
CREATE TABLE dq_alignment (
  basis TEXT, series_id TEXT, days INTEGER, exact INTEGER, carried INTEGER, missing INTEGER,
  max_staleness INTEGER, note TEXT, pct_exact REAL, pct_carried REAL, pct_missing REAL,
  PRIMARY KEY (basis, series_id));
CREATE TABLE dq_fx_crosscheck (
  date_key INTEGER PRIMARY KEY, rbi_ref REAL, fred_h10 REAL, status TEXT, diff_inr REAL,
  diff_pct REAL, large_gap_flag INTEGER);
CREATE TABLE dq_summary (
  metric TEXT PRIMARY KEY, value REAL, unit TEXT, explanation TEXT);

CREATE INDEX ix_fx_series ON fact_fx(series_id);
CREATE INDEX ix_ew_event ON fact_event_window(event_id, series_id);
CREATE INDEX ix_path_event ON fact_event_path(event_id, series_id);
"""

In [3]:
VIEWS = """
CREATE VIEW v_event_usdinr_summary AS
SELECT e.event_id, e.event_label, w.window_id, w.window_label, w.sort_order,
       f.basis, f.status, f.start_date, f.end_date, f.change AS usdinr_pct_change,
       f.inr_pct_change, f.usdinr_minus_dxy_pp, f.percentile, f.z_score,
       c.overlapping_events, c.rbi_repo_changes, c.fomc_target_changes
FROM fact_event_window f
JOIN dim_event e  ON e.event_id = f.event_id
JOIN dim_window w ON w.window_id = f.window_id
LEFT JOIN fact_event_window_context c
       ON c.basis = f.basis AND c.event_id = f.event_id AND c.window_id = f.window_id
WHERE f.series_id = 'USDINR';

CREATE VIEW v_event_cross_asset AS
SELECT f.basis, f.event_id, f.window_id, f.series_id, f.change, f.change_unit, f.status
FROM fact_event_window f;
"""

## Helpers: date keys, native-frequency facts, coverage and extra quality checks

In [4]:
def dkey(s):
    return pd.to_datetime(s).dt.strftime("%Y%m%d").astype(int)


def native_facts(obs):
    """Facts at each series' NATIVE frequency (no alignment) - for source-faithful charts."""
    obs = obs[obs.obs_date >= pd.Timestamp(ANALYSIS_START)].copy()
    out = {}

    def rets(g, level=False):
        g = g.sort_values("obs_date").copy()
        g["days_since_prev_obs"] = g["obs_date"].diff().dt.days
        if level:
            g["change"] = g["value"].diff()
            g["log_return"] = np.nan
        else:
            g["pct_change"] = (g["value"] / g["value"].shift(1) - 1) * 100
            g["log_return"] = np.log(g["value"] / g["value"].shift(1))
        return g

    fx = []
    for sid in ["USDINR_RBI", "USDINR_FRED", "USDINR_RBI_IBTABLE"]:
        g = rets(obs[obs.series_id == sid])
        g = g.rename(columns={"value": "usd_inr", "pct_change": "usdinr_pct_change"})
        g["inr_pct_change"] = (g["usd_inr"].shift(1) / g["usd_inr"] - 1) * 100
        fx.append(g)
    fx = pd.concat(fx)
    rbi_extra = obs[obs.series_id.isin(["USDINR_RBI_OPEN", "USDINR_RBI_CLOSE",
                                        "USDINR_RBI_INRSTRONG", "USDINR_RBI_INRWEAK"])]
    rbi_extra = rbi_extra.pivot_table(index="obs_date", columns="series_id", values="value")
    rbi_extra.columns = [{"USDINR_RBI_OPEN": "usdinr_open", "USDINR_RBI_CLOSE": "usdinr_close",
                          "USDINR_RBI_INRSTRONG": "inr_strongest",
                          "USDINR_RBI_INRWEAK": "inr_weakest"}[c] for c in rbi_extra.columns]
    fx = fx.merge(rbi_extra.reset_index(), on="obs_date", how="left")
    fx.loc[fx.series_id != "USDINR_RBI", ["usdinr_open", "usdinr_close", "inr_strongest", "inr_weakest"]] = np.nan
    fx["date_key"] = dkey(fx["obs_date"])
    out["fact_fx"] = fx[["date_key", "series_id", "usd_inr", "usdinr_open", "usdinr_close",
                         "inr_strongest", "inr_weakest", "usdinr_pct_change", "inr_pct_change",
                         "log_return", "days_since_prev_obs"]]

    com = pd.concat([rets(obs[obs.series_id == s]) for s in ["BRENT", "GOLD_USD"]])
    com["date_key"] = dkey(com["obs_date"])
    out["fact_commodity"] = com[["date_key", "series_id", "value", "pct_change", "log_return",
                                 "days_since_prev_obs"]]

    mk = []
    for s in ["NIFTY50", "DXY_BROAD", "VIX", "UST10Y"]:
        g = obs[obs.series_id == s]
        if g.empty:
            continue
        lvl = s == "UST10Y"
        g = rets(g, level=lvl)
        if not lvl:
            g["change"] = g["pct_change"]
        g["change_unit"] = "pp" if lvl else "%"
        mk.append(g)
    mk = pd.concat(mk)
    mk["date_key"] = dkey(mk["obs_date"])
    out["fact_market"] = mk[["date_key", "series_id", "value", "change", "change_unit",
                             "log_return", "days_since_prev_obs"]]
    return out

In [5]:
def coverage(obs):
    freq = {"US_CPI": "M", "IN_CPI_OECD": "M", "IN_CPI_MOSPI": "M", "IN_FX_RESERVES": "M",
            "IN_RBI_FX_NET": "M", "IN_RBI_FX_PURCHASE": "M", "IN_RBI_FX_SALE": "M", "IN_RBI_FWD_BOOK": "M",
            "IN_REPO": "event",
            "FFR_UPPER": "D (7-day)", "FFR_EFFECTIVE": "D (7-day)"}
    rows = []
    build = pd.Timestamp(ANALYSIS_END)
    for sid, g in obs.groupby("series_id"):
        d = g["obs_date"].sort_values()
        d = d[d >= pd.Timestamp(ANALYSIS_START)] if sid != "IN_REPO" else d
        gaps = d.diff().dt.days
        f = freq.get(sid, "D (business)")
        behind = (build - d.max()).days
        limit = {"M": 100, "event": 10_000}.get(f, 10)   # monthly data are published with a lag
        rows.append(dict(series_id=sid, first_date=d.min().date(), last_date=d.max().date(),
                         n_obs=len(d), expected_frequency=f,
                         max_gap_calendar_days=int(gaps.max()) if len(d) > 1 else None,
                         gaps_over_5d=int((gaps > 5).sum()) if f.startswith("D") else None,
                         days_behind_build_date=behind,
                         status=("STALE / ENDS EARLY" if behind > limit else
                                 "HAS GAPS > 10 DAYS" if f.startswith("D") and len(d) > 1 and gaps.max() > 10
                                 else "OK")))
    rows.append(dict(series_id="NIFTY50", first_date=None, last_date=None, n_obs=0,
                     expected_frequency="D (business)", status="MISSING - pending manual download")
                if "NIFTY50" not in set(obs.series_id) else {})
    return pd.DataFrame([r for r in rows if r])


def extra_checks(obs, xcheck, ew):
    iss = []
    for sid, thr in [("USDINR_RBI", 1.5), ("USDINR_FRED", 1.5), ("BRENT", 10), ("GOLD_USD", 5)]:
        g = obs[(obs.series_id == sid) & (obs.obs_date >= pd.Timestamp(ANALYSIS_START))].sort_values("obs_date")
        r = (g["value"] / g["value"].shift(1) - 1) * 100
        for d, v in zip(g["obs_date"][r.abs() > thr], r[r.abs() > thr]):
            iss.append(dict(stage="validate", series_id=sid, obs_date=d.date(), rule="large_daily_move",
                            detail=f"{v:+.2f}% day-on-day (threshold {thr}%)", severity="info",
                            action="kept - verified as market move, not a data error (review list)"))
    big = xcheck[xcheck.large_gap_flag == 1]
    iss.append(dict(stage="validate", series_id="USDINR_RBI vs USDINR_FRED", obs_date=None,
                    rule="source_divergence",
                    detail=f"{len(big)} days where |FRED-RBI| > 0.5% (different fixing times: ~13:30 IST vs 12:00 New York)",
                    severity="info", action="not corrected; RBI stays primary, FRED is a cross-check"))
    st = ew[(ew.series_id != "USDINR") & ((ew.start_staleness_days > 0) | (ew.end_staleness_days > 0))]
    iss.append(dict(stage="validate", series_id="event windows", obs_date=None,
                    rule="stale_window_endpoint",
                    detail=f"{len(st)} event-window rows use a carried-forward value at an endpoint (<= {MAX_STALENESS_DAYS} days old)",
                    severity="info", action="flag kept in fact_event_window.start/end_staleness_days"))
    return iss

## Build

The database is written to a temporary folder first and then copied into `02_Database/`. SQLite needs to create and
delete a journal file while writing, and synced or restricted folders sometimes block that.

In [6]:
def build_database():
    PBI.mkdir(parents=True, exist_ok=True)
    DB_PATH.parent.mkdir(parents=True, exist_ok=True)
    obs = pd.read_csv(PROC / "observations_long.csv", parse_dates=["obs_date"])
    t = {}

    # ---- dimensions
    t["dim_date"] = pd.read_csv(PROC / "dim_date.csv")
    t["dim_series"] = pd.read_csv(REF / REF_CATALOG)
    t["dim_fx_basis"] = pd.DataFrame([
        dict(basis="RBI", fx_series_id="USDINR_RBI", is_primary=1,
             calendar_definition="Days with an RBI/FBIL reference rate (Indian FX trading days)"),
        dict(basis="FRED", fx_series_id="USDINR_FRED", is_primary=0,
             calendar_definition="Days with a Fed H.10 USD/INR noon rate (US business days)")])
    ev = read_reference(REF_EVENTS)
    ev["duration_days"] = (pd.to_datetime(ev.event_end_date) - pd.to_datetime(ev.event_start_date)).dt.days + 1
    ev["event_label"] = ev["event_id"] + " · " + ev["short_name"] + " (" \
        + pd.to_datetime(ev["event_start_date"]).dt.strftime("%b-%Y") + ")"
    t["dim_event"] = ev
    ms = read_reference(REF_MILESTONES)
    ms["date_key"] = dkey(ms["milestone_date"])
    t["dim_event_milestone"] = ms
    t["dim_window"] = pd.DataFrame(
        [dict(window_id=w[0], window_label=w[1], start_offset=w[2], end_offset=w[3], window_type=w[4],
              sort_order=w[5],
              definition=f"Change from t({w[2]:+d}) to t({w[3]:+d}) in trading days; t(0)=first trading day on/after anchor date")
         for w in WINDOWS])

    # ---- native-frequency facts
    t.update(native_facts(obs))
    mm = pd.read_csv(PROC / "macro_monthly.csv", parse_dates=["month"])
    mm["date_key"] = dkey(mm["month"])
    mm["period_start"] = mm["month"].dt.strftime("%Y-%m-%d")
    mm["frequency"] = "M"
    t["fact_macro"] = mm[["date_key", "series_id", "period_start", "frequency", "value", "yoy_pct",
                          "mom_change", "series_version"]].dropna(subset=["value"])

    # ---- aligned daily panel (both bases)
    cols = {"USDINR": "usdinr", "USDINR_pct": "usdinr_pct", "INR_pct": "inr_pct",
            "USDINR_ret": "usdinr_logret", "BRENT": "brent", "GOLD_USD": "gold_usd",
            "NIFTY50": "nifty50", "DXY_BROAD": "dxy_broad", "VIX": "vix", "UST10Y": "ust10y",
            "FFR_UPPER": "ffr_upper", "IN_REPO": "in_repo", "BRENT_ret": "brent_logret",
            "GOLD_USD_ret": "gold_logret", "NIFTY50_ret": "nifty_logret", "DXY_BROAD_ret": "dxy_logret",
            "VIX_ret": "vix_logret", "UST10Y_ret": "ust10y_change_pp", "USDINR_vol20": "usdinr_vol20",
            "BRENT_vol20": "brent_vol20", "GOLD_USD_vol20": "gold_vol20", "NIFTY50_vol20": "nifty_vol20",
            "DXY_BROAD_vol20": "dxy_vol20", "corr60_USDINR_BRENT": "corr60_brent",
            "corr60_USDINR_GOLD_USD": "corr60_gold", "corr60_USDINR_NIFTY50": "corr60_nifty",
            "corr60_USDINR_DXY_BROAD": "corr60_dxy", "corr60_USDINR_VIX": "corr60_vix",
            "corr60_USDINR_UST10Y": "corr60_ust10y", "basis": "basis", "trading_day_seq": "trading_day_seq"}
    panels = []
    for b in FX_SOURCES:
        p = pd.read_csv(PROC / f"panel_wide_{b}.csv", parse_dates=["date"])
        p = p.rename(columns=cols)
        p["date_key"] = dkey(p["date"])
        p["rate_diff_in_us"] = p["in_repo"] - p["ffr_upper"]
        panels.append(p)
    ddl_cols = ["basis", "date_key", "trading_day_seq", "usdinr", "usdinr_pct", "inr_pct", "usdinr_logret",
                "brent", "gold_usd", "nifty50", "dxy_broad", "vix", "ust10y", "ffr_upper", "in_repo",
                "rate_diff_in_us", "brent_logret", "gold_logret", "nifty_logret", "dxy_logret",
                "vix_logret", "ust10y_change_pp", "usdinr_vol20", "brent_vol20", "gold_vol20",
                "nifty_vol20", "dxy_vol20", "corr60_brent", "corr60_gold", "corr60_nifty", "corr60_dxy",
                "corr60_vix", "corr60_ust10y"]
    t["fact_daily_aligned"] = pd.concat(panels)[ddl_cols]

    # ---- event facts
    an = pd.read_csv(PROC / "event_anchor.csv")
    an["t0_date_key"] = pd.to_datetime(an["t0_date"]).dt.strftime("%Y%m%d").astype("Int64")
    t["fact_event_anchor"] = an
    t["dim_analysis_series"] = pd.DataFrame([
        ("USDINR", "USD/INR", "INR per USD", "%", 1, "USDINR_RBI", "USDINR_FRED", "USD buys more rupees = INR weakened"),
        ("BRENT", "Brent crude", "USD/bbl", "%", 2, "BRENT", "BRENT", "Oil more expensive"),
        ("GOLD_USD", "Gold (LBMA AM)", "USD/oz", "%", 3, "GOLD_USD", "GOLD_USD", "Gold more expensive"),
        ("NIFTY50", "NIFTY 50", "Index", "%", 4, "NIFTY50", "NIFTY50", "Indian equities up"),
        ("DXY_BROAD", "Broad US dollar index", "Index", "%", 5, "DXY_BROAD", "DXY_BROAD", "USD stronger vs 26 currencies"),
        ("VIX", "VIX", "Index", "%", 6, "VIX", "VIX", "Higher US equity implied volatility (risk-off)"),
        ("UST10Y", "US 10Y yield", "%", "pp", 7, "UST10Y", "UST10Y", "Higher US long rates"),
        ("FFR_UPPER", "Fed funds target (upper)", "%", "pp", 8, "FFR_UPPER", "FFR_UPPER", "Fed tightened"),
        ("IN_REPO", "RBI repo rate", "%", "pp", 9, "IN_REPO", "IN_REPO", "RBI tightened"),
    ], columns=["series_id", "display_name", "unit", "change_unit", "sort_order",
                "source_series_rbi_basis", "source_series_fred_basis", "interpretation_of_positive"])
    ew = pd.read_csv(PROC / "event_window.csv")
    ew["direction"] = np.select(
        [ew.series_id.eq("USDINR") & (ew.change > 0), ew.series_id.eq("USDINR") & (ew.change < 0)],
        ["INR weakened", "INR strengthened"],
        np.where(ew.change > 0, "up", np.where(ew.change < 0, "down", None)))
    ew["is_unusual_2sd"] = np.where(ew["z_score"].notna(), (ew["z_score"].abs() >= 2).astype(int), np.nan)
    t["fact_event_window"] = ew
    t["fact_event_window_corr"] = pd.read_csv(PROC / "event_window_corr.csv")
    t["fact_event_path"] = pd.read_csv(PROC / "event_path.csv").dropna(subset=["indexed_value"])
    ctx = pd.read_csv(PROC / "event_window_context.csv")
    txt = ["overlapping_events", "rbi_repo_changes", "fomc_target_changes", "rbi_fx_months"]
    ctx[txt] = ctx[txt].fillna("")
    ctx["n_overlapping"] = ctx["overlapping_events"].apply(lambda s: len([x for x in s.split(",") if x]))
    ctx["has_policy_change"] = ((ctx.rbi_repo_changes != "") | (ctx.fomc_target_changes != "")).astype(int)
    t["fact_event_window_context"] = ctx

    # ---- data quality
    xc = pd.read_csv(PROC / "fx_crosscheck.csv", parse_dates=["date"])
    issues = pd.read_csv(PROC / "dq_issues_ingest.csv")
    issues = pd.concat([issues, pd.DataFrame(extra_checks(obs, xc, ew))], ignore_index=True)
    issues.insert(0, "issue_id", range(1, len(issues) + 1))
    t["dq_issues"] = issues
    t["dq_series_coverage"] = coverage(obs)
    al = pd.read_csv(PROC / "dq_alignment.csv")
    for c in ["exact", "carried", "missing"]:
        al[f"pct_{c}"] = al[c] / al["days"].replace(0, np.nan) * 100
    t["dq_alignment"] = al
    xc["date_key"] = dkey(xc["date"])
    t["dq_fx_crosscheck"] = xc.drop(columns="date")
    both = xc[xc.status == "both"].set_index("date")
    r = np.log(both[["rbi_ref", "fred_h10"]]).diff().dropna()
    wk = np.log(both[["rbi_ref", "fred_h10"]].resample("W-FRI").last()).diff().dropna()
    t["dq_summary"] = pd.DataFrame([
        dict(metric="fx_days_both_sources", value=len(both), unit="days",
             explanation="Days where both RBI reference rate and Fed H.10 rate exist"),
        dict(metric="fx_mean_abs_diff_pct", value=both.diff_pct.abs().mean(), unit="%",
             explanation="Average |FRED - RBI| / RBI"),
        dict(metric="fx_daily_return_corr", value=r.corr().iloc[0, 1], unit="corr",
             explanation="Correlation of DAILY log returns - lowered by the ~9h timestamp gap"),
        dict(metric="fx_weekly_return_corr", value=wk.corr().iloc[0, 1], unit="corr",
             explanation="Correlation of WEEKLY (Fri) log returns - timestamp noise mostly washes out"),
        dict(metric="rbi_last_date", value=int(obs[obs.series_id == "USDINR_RBI"].obs_date.max().strftime("%Y%m%d")),
             unit="yyyymmdd", explanation="RBI DBIE export ends here; later events use FRED basis"),
        dict(metric="events_total", value=len(ev), unit="count", explanation="Events in dim_event"),
        dict(metric="event_windows_complete_rbi",
             value=int(((ew.basis == "RBI") & (ew.series_id == "USDINR") & (ew.status == "complete")).sum()),
             unit="count", explanation="USD/INR event windows with both endpoints on the RBI calendar"),
        dict(metric="event_windows_with_overlap", value=int((ctx.n_overlapping > 0).sum()), unit="count",
             explanation="Event windows that contain another event's anchor day (all bases)"),
        dict(metric="event_windows_with_policy_change", value=int(ctx.has_policy_change.sum()), unit="count",
             explanation="Event windows containing an RBI repo change or FOMC target change (all bases)"),
        dict(metric="nifty_loaded", value=int("NIFTY50" in set(obs.series_id)), unit="flag",
             explanation="1 once Nifty CSVs are in 01_Data/1_Raw/NSE_Nifty50 and the pipeline is re-run"),
    ])

    # ---- write SQLite
    # Build in a temp folder, then copy into 02_Database/ in one go. SQLite needs to create
    # and delete a journal file while writing; some synced / network / sandboxed
    # folders do not allow that, so the project folder only ever receives a copy.
    tmp_db = Path(tempfile.mkdtemp()) / DB_PATH.name
    con = sqlite3.connect(tmp_db)
    con.executescript(DDL)
    for name, df in t.items():
        cols_db = [r[1] for r in con.execute(f"PRAGMA table_info({name})")]
        df = df.copy()
        for c in cols_db:
            if c not in df.columns:
                df[c] = None
        try:
            df[cols_db].to_sql(name, con, if_exists="append", index=False)
        except Exception as exc:
            raise RuntimeError(f"load failed for {name}") from exc
        df[cols_db].to_csv(PBI / f"{name}.csv", index=False)
    con.executescript(VIEWS)
    con.commit()
    schema = "\n".join(r[0] + ";" for r in con.execute(
        "SELECT sql FROM sqlite_master WHERE sql IS NOT NULL ORDER BY type DESC, name"))
    (DB_PATH.parent / "schema.sql").write_text(schema)
    for name in t:
        print(f"{name:28s}", con.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0])
    con.close()
    journal = DB_PATH.with_name(DB_PATH.name + "-journal")
    if journal.exists():                       # left over from an interrupted write
        try:
            journal.unlink()
        except OSError:
            open(journal, "wb").close()        # a 0-byte journal is ignored by SQLite
    shutil.copyfile(tmp_db, DB_PATH)
    shutil.rmtree(tmp_db.parent, ignore_errors=True)
    print("database written to", DB_PATH)

In [7]:
build_database()

dim_date                     5021
dim_series                   23
dim_fx_basis                 2
dim_event                    13
dim_event_milestone          26
dim_window                   8
fact_fx                      9997
fact_commodity               6954
fact_market                  13749
fact_macro                   1784
fact_daily_aligned           6749
fact_event_anchor            26
dim_analysis_series          9
fact_event_window            1872
fact_event_window_corr       624
fact_event_path              27709
fact_event_window_context    208
dq_issues                    119
dq_series_coverage           24
dq_alignment                 16
dq_fx_crosscheck             3569
dq_summary                   10


database written to /sessions/rcw-01qt75kwf65eog3edxywzaqj/mnt/DA PROJECTS/Geopolitical Events & INRUSD/02_Database/geo_inr_analytics.db


## Check the result

In [8]:
import sqlite3
con = sqlite3.connect(DB_PATH)
pd.read_sql("SELECT metric, ROUND(value, 4) AS value, explanation FROM dq_summary", con)

,metric,value,explanation
0,fx_days_both_sources,3.180000e+03,Days where both RBI reference rate and Fed H.10 rate exist
1,fx_mean_abs_diff_pct,1.738000e-01,Average |FRED - RBI| / RBI
2,fx_daily_return_corr,5.643000e-01,Correlation of DAILY log returns - lowered by the ~9h timestamp gap
3,fx_weekly_return_corr,8.914000e-01,Correlation of WEEKLY (Fri) log returns - timestamp noise mostly washes out
4,rbi_last_date,2.026093e+07,RBI DBIE export ends here; later events use FRED basis
5,events_total,1.300000e+01,Events in dim_event
6,event_windows_complete_rbi,1.030000e+02,USD/INR event windows with both endpoints on the RBI calendar
7,event_windows_with_overlap,4.000000e+00,Event windows that contain another event's anchor day (all bases)
8,event_windows_with_policy_change,5.300000e+01,Event windows containing an RBI repo change or FOMC target change (all bases)
9,nifty_loaded,1.000000e+00,1 once Nifty CSVs are in 01_Data/1_Raw/NSE_Nifty50 and the pipeline is re-run


In [9]:
pd.read_sql("""
SELECT event_label, window_label, ROUND(usdinr_pct_change, 2) AS usdinr_pct,
       ROUND(percentile, 1) AS pctile_vs_prior_3y, rbi_repo_changes, overlapping_events
FROM v_event_usdinr_summary
WHERE basis = 'RBI' AND window_id = 'POST_30'
ORDER BY event_id""", con)

,event_label,window_label,usdinr_pct,pctile_vs_prior_3y,rbi_repo_changes,overlapping_events
0,E01 · Crimea annexation (Mar-2014),30 days after,-2.19,8.3,,
1,E02 · Surgical strikes after Uri (Sep-2016),30 days after,2.00,80.3,2016-10-04 -25bp,
2,E03 · Doklam standoff (Jun-2017),30 days after,-0.47,29.3,2017-08-02 -25bp,
3,E04 · Pulwama & Balakot (Feb-2019),30 days after,-2.95,7.5,2019-04-04 -25bp,
4,E05 · Soleimani strike (Jan-2020),30 days after,0.06,48.3,,
5,E06 · Galwan Valley clash (Jun-2020),30 days after,-1.79,10.1,,
6,E07 · Russia invades Ukraine (Feb-2022),30 days after,2.00,85.9,,
7,E08 · Hamas attack & Gaza war (Oct-2023),30 days after,0.12,43.5,,
8,E09 · Iran strikes Israel (Apr-2024),30 days after,-0.11,31.8,,
9,E10 · Pahalgam & Op Sindoor (Apr-2025),30 days after,0.81,67.6,2025-06-06 -50bp,


In [10]:
pd.read_sql("SELECT series_id, first_date, last_date, n_obs, max_gap_calendar_days, status FROM dq_series_coverage", con)

,series_id,first_date,last_date,n_obs,max_gap_calendar_days,status
0,BRENT,2013-01-02,2026-09-22,3481,6,OK
1,DXY_BROAD,2013-01-02,2026-09-25,3421,6,OK
2,FFR_EFFECTIVE,2013-01-01,2026-09-28,5019,1,OK
3,FFR_UPPER,2013-01-01,2026-09-29,5020,1,OK
4,GOLD_USD,2013-01-02,2026-09-29,3473,5,OK
5,IN_CPI_MOSPI,2013-01-01,2026-08-01,164,31,OK
6,IN_CPI_OECD,2013-01-01,2025-03-01,147,31,STALE / ENDS EARLY
7,IN_FX_RESERVES,2013-01-01,2026-07-01,163,31,OK
8,IN_RBI_FWD_BOOK,2013-01-01,2026-07-01,163,31,OK
9,IN_RBI_FX_NET,2013-01-01,2026-07-01,163,31,OK


In [11]:
con.close()

**Output:** `02_Database/geo_inr_analytics.db`, `schema.sql`, and `01_Data/4_PowerBI_Tables/*.csv`. Next: **05 · Results report**.